[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_07_gans_generative_adversarial_networks/aula_07_gans_generative_adversarial_networks.ipynb)

# 🔬 Aula 07: Redes Generativas (cGAN & CycleGAN) para Resolução de Escassez de Dados em Visão Computacional Biomédica
### **Faculdade Infnet — Pós-Graduação em Visão Computacional com CNNs e Transformers**
**Professor:** Dr. Allan Spadini  
**Metodologia Pedagógica Central:** Situação-Problema do Mundo Real ➔ Solução de Engenharia ➔ Teoria Rigorosa

---

### 🎯 Objetivos Deste Laboratório:
1. **Auditoria Diagnóstica de Dados Biomédicos:** Carregar e inspecionar um dataset de imagens de microscopia/diagnóstico celular, diagnosticando o impacto do **desbalanceamento severo de classes** (fenótipo comum vs patologia/marcador raro).
2. **Classificador Baseline:** Treinar uma ResNet-18 no dataset desbalanceado original e mensurar a fragilidade do **Recall / Sensibilidade** (proliferação de Falsos Negativos diagnósticos).
3. **Modelagem Generativa com GAN Condicional (cGAN):** Projetar e implementar do zero em PyTorch uma cGAN com condicionamento estrutural via embedding para geração direcionada da classe minoritária escassa (inspirado no Project 9B: Virtual Staining).
4. **Treinamento Otimizado para Google Colab T4:** Treinamento adversarial estável com heurística não-saturante ($-\log D$), desacoplamento com `.detach()` e estabilidade de gradientes.
5. **Data Augmentation Generativo:** Expandir a base de treino sintetizando amostras da classe rara e retreinar o classificador downstream.
6. **Avaliação Comparativa de Recall:** Comprovar experimentalmente o salto no **Recall da classe minoritária** em um conjunto de teste 100% real, reduzindo drasticamente os Falsos Negativos.
7. **Arquitetura de Tradução Não-Pareada (CycleGAN):** Discutir e estruturar a formulação de tradução de domínio óptico sem pares alinhados com consistência de ciclo $\mathcal{L}_{cyc}$ (Project 9C: Holo2Bright).


---
## ⚙️ Parte 1: Configuração do Ambiente, GPU e Download Programático

Configuramos o ambiente do Colab com GPU T4, fixamos sementes para reprodutibilidade científica e realizamos o download programático do dataset biomédico.


In [ ]:
# Instalação das bibliotecas necessárias no Google Colab
!pip install -q kagglehub torchvision scikit-learn matplotlib seaborn tqdm


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import torchvision.transforms as T
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import confusion_matrix, classification_report, recall_score, precision_score, f1_score

# 1. Reprodutibilidade científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Detecção automática de acelerador CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Dispositivo Ativo: {device}")
if torch.cuda.is_available():
    print(f"🚀 GPU Detectada: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM Total Disponível: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ Atenção: Executando em CPU. Para acelerar o treino, ative a GPU T4 em Ambiente de Execução.")


### Download Automático do Dataset Biomédico
Utilizamos um conjunto de dados biomédicos de patologia/citologia celular para simular a classificação entre:
- **Classe 0 (Controle / Normal):** Fenótipo celular comum, abundante em coletas de rotina.
- **Classe 1 (Patologia / Marcador Raro):** Fenótipo crítico escasso, com alta relevância diagnóstica.
Caso o download online via `kagglehub` encontre restrições de rede, nosso script contém um gerador de fallback sintético estruturado para garantir execução 100% contínua e sem quebras.


In [ ]:
import kagglehub

dataset_root = "./biomedical_cells_dataset"

try:
    download_path = kagglehub.dataset_download("paultimothymooney/blood-cells")
    # Localizar pasta com imagens de treino e teste
    cand_train = os.path.join(download_path, "dataset2-master", "dataset2-master", "images", "TRAIN")
    cand_test = os.path.join(download_path, "dataset2-master", "dataset2-master", "images", "TEST")
    if os.path.exists(cand_train) and os.path.exists(cand_test):
        dataset_root = os.path.join(download_path, "dataset2-master", "dataset2-master", "images")
        print(f"✅ Dataset biomédico carregado com sucesso em: {dataset_root}")
    else:
        raise ValueError("Estrutura padrão de pastas não encontrada.")
except Exception as e:
    print(f"ℹ️ Configurando diretório biomédico estruturado local ({e})...")
    os.makedirs(os.path.join(dataset_root, "TRAIN", "Common_Normal"), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, "TRAIN", "Rare_Pathology"), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, "TEST", "Common_Normal"), exist_ok=True)
    os.makedirs(os.path.join(dataset_root, "TEST", "Rare_Pathology"), exist_ok=True)
    
    # Gerar amostras microscópicas de demonstração (células esféricas e núcleos celulares)
    for split, count in [("TRAIN", 60), ("TEST", 20)]:
        for cls, is_rare in [("Common_Normal", False), ("Rare_Pathology", True)]:
            for i in range(count):
                img_arr = np.ones((64, 64), dtype=np.uint8) * 40
                # Criar morfologia celular sintética
                yy, xx = np.ogrid[:64, :64]
                center_y, center_x = 32 + np.random.randint(-4, 5), 32 + np.random.randint(-4, 5)
                radius = 18 if not is_rare else 22
                cell_mask = (xx - center_x)**2 + (yy - center_y)**2 < radius**2
                img_arr[cell_mask] = 160
                
                # Adicionar núcleos (mais densos e hipertrofiados na classe rara)
                nuc_radius = 8 if not is_rare else 14
                nuc_mask = (xx - center_x)**2 + (yy - center_y)**2 < nuc_radius**2
                img_arr[nuc_mask] = 230 if is_rare else 90
                
                # Ruído óptico de microscopia
                noise = np.random.normal(0, 10, (64, 64)).astype(np.int16)
                img_arr = np.clip(img_arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
                Image.fromarray(img_arr).save(os.path.join(dataset_root, split, cls, f"cell_{i}.png"))
    print(f"✅ Conjunto celular configurado com sucesso em: {dataset_root}")


---
## 📊 Parte 2: Exploração de Dados e Simulação do Desbalanceamento Diagnóstico

Em triagens e análises laboratoriais, fenótipos raros ou patologias críticas são altamente escassos quando comparados a amostras normais de controle.
Para simular com fidelidade o **desafio prático do mundo real**, criamos um particionamento de treinamento desbalanceado:
- **Classe Comum / Normal (Majoritária):** 100% das amostras normais disponíveis.
- **Classe Rara / Patológica (Minoritária):** Apenas 20% das amostras reais para simular severa escassez.
- **Conjunto de Teste:** Preservado intacto e composto **exclusivamente por dados 100% reais** para auditoria justa.


In [ ]:
from torchvision.datasets import ImageFolder

IMG_SIZE = 64 # Resolução otimizada para treino rápido e estável no Colab T4
BATCH_SIZE = 32

# Transformações de imagem
train_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=1), # Monocromático para microscopia de contraste / fase
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5]) # Normalização [-1, 1] compatível com Tanh em GANs
])

test_transforms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

# Localizar diretórios de treino e teste
train_dir = os.path.join(dataset_root, "TRAIN") if os.path.exists(os.path.join(dataset_root, "TRAIN")) else os.path.join(dataset_root, "train")
test_dir = os.path.join(dataset_root, "TEST") if os.path.exists(os.path.join(dataset_root, "TEST")) else os.path.join(dataset_root, "test")

full_train_dataset = ImageFolder(train_dir, transform=train_transforms)
full_test_dataset = ImageFolder(test_dir, transform=test_transforms)

class_names = full_train_dataset.classes
print(f"🏷️ Classes encontradas no dataset: {class_names}")

# Identificar classe de referência (comum/normal) e classe rara (patológica/minoritária)
normal_idx = 0
rare_idx = 1 if len(class_names) > 1 else 0

train_indices_normal = [i for i, (_, label) in enumerate(full_train_dataset.samples) if label == normal_idx]
train_indices_rare = [i for i, (_, label) in enumerate(full_train_dataset.samples) if label == rare_idx]
test_indices = [i for i, (_, label) in enumerate(full_test_dataset.samples) if label in [normal_idx, rare_idx]]

# Simulação da escassez: reter 100% da classe comum e apenas 20% da classe rara no treino
num_scarce_rare = max(5, int(len(train_indices_rare) * 0.20))
scarce_rare_indices = train_indices_rare[:num_scarce_rare]

imbalanced_train_indices = train_indices_normal + scarce_rare_indices
imbalanced_train_set = Subset(full_train_dataset, imbalanced_train_indices)
test_set = Subset(full_test_dataset, test_indices)

print(f"📊 Distribuição de Treino Desbalanceado:")
print(f"   - Classe Comum / Normal (Majoritária): {len(train_indices_normal)} imagens")
print(f"   - Classe Rara / Patológica (Escassa):    {len(scarce_rare_indices)} imagens")
print(f"   - Total no Conjunto de Treino:          {len(imbalanced_train_set)} imagens")
print(f"🧪 Teste de Avaliação 100% Real:           {len(test_set)} imagens")


In [ ]:
# Visualização das Amostras Reais de Microscopia Celular
def denormalize(tensor):
    return (tensor * 0.5 + 0.5).clamp(0, 1)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle("Amostras Reais do Dataset de Microscopia Celular", fontsize=14, fontweight="bold")

for idx, ax in enumerate(axes):
    sample_idx = imbalanced_train_indices[idx if idx < 2 else -idx]
    img, lbl = full_train_dataset[sample_idx]
    ax.imshow(denormalize(img).squeeze(), cmap="bone")
    lbl_name = "Classe Rara (1)" if lbl == rare_idx else "Comum/Normal (0)"
    ax.set_title(f"{lbl_name}", color="crimson" if lbl == rare_idx else "darkgreen", fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## 🏥 Parte 3: Treinamento do Classificador Baseline e Auditoria de Recall

Treinamos um classificador convolucional profundo (ResNet-18 adaptado) na base desbalanceada.
O objetivo é demonstrar a **dor prática do mundo real**:
> **O Paradoxo da Acurácia:** O classificador atinge acurácia global moderada ou alta, mas falha gravemente no **Recall (Sensibilidade)** da classe minoritária rara, gerando uma taxa inaceitável de **Falsos Negativos**.


In [ ]:
# Construção do Classificador ResNet-18 Adaptado para Canal Único
def build_classifier(num_classes=2):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    original_conv = model.conv1
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        model.conv1.weight = nn.Parameter(original_conv.weight.mean(dim=1, keepdim=True))
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def train_classifier(model, train_loader, epochs=6, lr=1e-4):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for imgs, labels in train_loader:
            binary_labels = torch.tensor([1 if l == rare_idx else 0 for l in labels], device=device)
            imgs = imgs.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, binary_labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            
    return model

def evaluate_classifier(model, test_loader, title="Modelo"):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for imgs, labels in test_loader:
            binary_labels = [1 if l == rare_idx else 0 for l in labels]
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            
            all_preds.extend(preds)
            all_targets.extend(binary_labels)
            
    cm = confusion_matrix(all_targets, all_preds)
    rec = recall_score(all_targets, all_preds, pos_label=1, zero_division=0)
    prec = precision_score(all_targets, all_preds, pos_label=1, zero_division=0)
    f1 = f1_score(all_targets, all_preds, pos_label=1, zero_division=0)
    
    print(f"\n--- 📋 Avaliação Diagnóstica: {title} ---")
    print(f"Recall da Classe Rara (Sensibilidade): {rec * 100:.2f}%")
    print(f"Precisão da Classe Rara:              {prec * 100:.2f}%")
    print(f"F1-Score:                             {f1:.3f}")
    print(f"Matriz de Confusão (TN, FP / FN, TP):\n{cm}")
    
    return {"cm": cm, "recall": rec, "precision": prec, "f1": f1, "targets": all_targets, "preds": all_preds}


In [ ]:
# Treinamento do Modelo Baseline (Sem Dados Sintéticos)
baseline_loader = DataLoader(imbalanced_train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

print("⏳ Treinando Classificador Baseline no dataset desbalanceado...")
baseline_model = build_classifier()
baseline_model = train_classifier(baseline_model, baseline_loader, epochs=6)

print("🧪 Avaliando Baseline no teste 100% real...")
baseline_metrics = evaluate_classifier(baseline_model, test_loader, title="Baseline (Sem GAN)")


---
## 🧠 Parte 4: Implementação da GAN Condicional (cGAN) em PyTorch

Para combater a escassez da classe minoritária, implementamos uma **Conditional DCGAN (cGAN)** inspirada na formulação do Project 9B (Virtual Staining):
- **Gerador Condicional $G(z, y)$:** Recebe ruído $z \in \mathbb{R}^{100}$ e o rótulo da classe $y \in \{0, 1\}$. O rótulo é transformado por uma camada `nn.Embedding(2, 16)` e concatenado ao vetor latente, projetando $[B, 116, 1, 1] \to [B, 1, 64, 64]$.
- **Discriminador Condicional $D(x, y)$:** Recebe a imagem $[B, 1, 64, 64]$ concatenada com um mapa espacial replicado do embedding de classe $[B, 1, 64, 64]$, avaliando se a imagem é **real E se corresponde ao fenótipo solicitado**.


In [ ]:
# 1. Gerador Condicional (cGAN)
class ConditionalGenerator(nn.Module):
    def __init__(self, nz=100, num_classes=2, embed_dim=16, ngf=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        
        # Entrada combinada: [B, nz + embed_dim, 1, 1]
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz + embed_dim, ngf * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(ngf * 8, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(ngf, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )
        
    def forward(self, z, labels):
        c = self.label_emb(labels).unsqueeze(2).unsqueeze(3)
        x = torch.cat([z, c], dim=1)
        return self.net(x)

# 2. Discriminador Condicional (cGAN)
class ConditionalDiscriminator(nn.Module):
    def __init__(self, num_classes=2, img_size=64, ndf=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, img_size * img_size)
        self.img_size = img_size
        
        self.net = nn.Sequential(
            nn.Conv2d(2, ndf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid()
        )
        
    def forward(self, img, labels):
        c = self.label_emb(labels).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, c], dim=1)
        return self.net(x).view(-1, 1).squeeze(1)

# Inicialização de pesos (Radford et al., 2015)
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG = ConditionalGenerator().to(device)
netD = ConditionalDiscriminator().to(device)
netG.apply(weights_init)
netD.apply(weights_init)

print(f"✅ cGAN Instanciada com Sucesso no dispositivo {device}!")
print(f"   - Parâmetros do Gerador:     {sum(p.numel() for p in netG.parameters()):,}")
print(f"   - Parâmetros do Discriminador: {sum(p.numel() for p in netD.parameters()):,}")


---
## ⚡ Parte 5: Treinamento Adversarial Estável (Heurística Não-Saturante & `.detach()`)

Executamos o loop de treino da cGAN no dataset desbalanceado com técnicas de engenharia industrial:
1. **One-Sided Label Smoothing:** Rótulos reais = $0.9$ para evitar confiança excessiva do discriminador.
2. **Desacoplamento via `.detach()`:** Tratamento das amostras sintéticas como constantes puras durante a atualização de $D$.
3. **Heurística Não-Saturante:** Otimização de $-\log D$ para $G$ com rótulos $1.0$, prevenindo gradientes nulos no início do treino.


In [ ]:
criterion_bce = nn.BCELoss()
lr_gan = 2e-4
beta1 = 0.5

optimizerD = optim.Adam(netD.parameters(), lr=lr_gan, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr_gan, betas=(beta1, 0.999))

NZ = 100
EPOCHS_GAN = 12 # Ajustado para execução ágil na GPU T4
fixed_noise = torch.randn(16, NZ, 1, 1, device=device)
fixed_rare_labels = torch.ones(16, dtype=torch.long, device=device) # y = 1 (Classe Rara)

gan_loader = DataLoader(imbalanced_train_set, batch_size=BATCH_SIZE, shuffle=True)

print("🚀 Iniciando Treinamento Adversarial da cGAN...")
for epoch in range(EPOCHS_GAN):
    loss_d_epoch = 0.0
    loss_g_epoch = 0.0
    
    for imgs, labels in gan_loader:
        b_size = imgs.size(0)
        binary_labels = torch.tensor([1 if l == rare_idx else 0 for l in labels], device=device)
        real_imgs = imgs.to(device)
        
        # ----------------------------------------------------
        # Passo 1: Atualizar Discriminador D
        # ----------------------------------------------------
        optimizerD.zero_grad()
        
        # Perda com amostras reais (Label Smoothing = 0.9)
        real_targets = torch.full((b_size,), 0.9, device=device)
        out_real = netD(real_imgs, binary_labels)
        loss_d_real = criterion_bce(out_real, real_targets)
        
        # Perda com amostras falsas
        noise = torch.randn(b_size, NZ, 1, 1, device=device)
        fake_imgs = netG(noise, binary_labels)
        fake_targets = torch.zeros(b_size, device=device)
        
        # .detach() OBRIGATÓRIO: quebra o grafo para não retropropagar em G aqui!
        out_fake = netD(fake_imgs.detach(), binary_labels)
        loss_d_fake = criterion_bce(out_fake, fake_targets)
        
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        optimizerD.step()
        
        # ----------------------------------------------------
        # Passo 2: Atualizar Gerador G (Heurística Não-Saturante)
        # ----------------------------------------------------
        optimizerG.zero_grad()
        out_fake_for_g = netD(fake_imgs, binary_labels)
        g_targets = torch.ones(b_size, device=device)
        loss_g = criterion_bce(out_fake_for_g, g_targets)
        
        loss_g.backward()
        optimizerG.step()
        
        loss_d_epoch += loss_d.item()
        loss_g_epoch += loss_g.item()
        
    if (epoch + 1) % 3 == 0 or epoch == EPOCHS_GAN - 1:
        print(f"Época [{epoch+1}/{EPOCHS_GAN}] | Loss D: {loss_d_epoch/len(gan_loader):.4f} | Loss G: {loss_g_epoch/len(gan_loader):.4f}")

print("✅ Treinamento da cGAN Concluído com Sucesso!")


In [ ]:
# Visualização das Amostras Sintéticas da Classe Rara Geradas
netG.eval()
with torch.no_grad():
    synthetic_rare_imgs = netG(fixed_noise, fixed_rare_labels).cpu()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
fig.suptitle("Amostras Sintéticas da Classe Rara Geradas pela cGAN G(z, y=Rara)", fontsize=13, fontweight="bold")

for idx, ax in enumerate(axes.flat):
    img = denormalize(synthetic_rare_imgs[idx]).squeeze()
    ax.imshow(img, cmap="bone")
    ax.set_title(f"Sintético #{idx+1}", fontsize=10, color="crimson")
    ax.axis("off")

plt.tight_layout()
plt.show()


---
## 📈 Parte 6: Data Augmentation Sintética e Comparativo Experimental de Recall

Agora executamos o teste experimental decisivo:
1. Sintetizamos novas amostras da classe escassa com a cGAN treinada.
2. Criamos o dataset aumentado: **Dados Reais de Treino + Dados Sintéticos Gerados**.
3. Retreinamos a ResNet-18 sob hiperparâmetros idênticos.
4. Avaliamos o impacto no **Recall da classe minoritária** no conjunto de teste 100% real!


In [ ]:
# Dataset Customizado para Incorporar Amostras Sintéticas
class SyntheticDataset(Dataset):
    def __init__(self, tensors, labels):
        self.tensors = tensors
        self.labels = labels
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        return self.tensors[idx], self.labels[idx]

# Gerar 80 amostras sintéticas da classe rara
NUM_SYNTHETIC = 80
netG.eval()
with torch.no_grad():
    gen_noise = torch.randn(NUM_SYNTHETIC, NZ, 1, 1, device=device)
    gen_labels = torch.ones(NUM_SYNTHETIC, dtype=torch.long, device=device)
    synthetic_tensors = netG(gen_noise, gen_labels).cpu()

synthetic_dataset = SyntheticDataset(synthetic_tensors, [rare_idx] * NUM_SYNTHETIC)

# Concatenar dataset de treino original desbalanceado com dados sintéticos
augmented_train_set = ConcatDataset([imbalanced_train_set, synthetic_dataset])
augmented_loader = DataLoader(augmented_train_set, batch_size=BATCH_SIZE, shuffle=True)

print(f"📦 Dataset de Treino Original:    {len(imbalanced_train_set)} amostras")
print(f"✨ Amostras Sintéticas Geradas:    {NUM_SYNTHETIC} amostras da classe rara")
print(f"🎯 Dataset Aumentado para Treino: {len(augmented_train_set)} amostras")


In [ ]:
print("⏳ Treinando Classificador Aumentado com Dados Sintéticos...")
augmented_model = build_classifier()
augmented_model = train_classifier(augmented_model, augmented_loader, epochs=6)

print("🧪 Avaliando Classificador Aumentado no teste 100% real...")
augmented_metrics = evaluate_classifier(augmented_model, test_loader, title="Aumentado (Com cGAN)")


In [ ]:
# Comparação Visual e Quantitativa do Recall
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Comparativo das Matrizes de Confusão no Teste Real (Comum vs Rara)", fontsize=14, fontweight="bold")

# Matriz 1: Baseline
sns.heatmap(baseline_metrics["cm"], annot=True, fmt="d", cmap="Reds", ax=axes[0], cbar=False,
            xticklabels=["Comum", "Rara"], yticklabels=["Comum", "Rara"])
axes[0].set_title(f"Baseline (Sem GAN)\nRecall Rara: {baseline_metrics['recall']*100:.1f}%", fontweight="bold")
axes[0].set_ylabel("Classe Real")
axes[0].set_xlabel("Predição")

# Matriz 2: Aumentado
sns.heatmap(augmented_metrics["cm"], annot=True, fmt="d", cmap="Greens", ax=axes[1], cbar=False,
            xticklabels=["Comum", "Rara"], yticklabels=["Comum", "Rara"])
axes[1].set_title(f"Aumentado (Com cGAN)\nRecall Rara: {augmented_metrics['recall']*100:.1f}%", fontweight="bold", color="darkgreen")
axes[1].set_ylabel("Classe Real")
axes[1].set_xlabel("Predição")

plt.tight_layout()
plt.show()

# Gráfico de Barras das Métricas Diagnósticas
metrics_names = ["Recall (Sensibilidade)", "F1-Score", "Precisão"]
base_vals = [baseline_metrics["recall"]*100, baseline_metrics["f1"]*100, baseline_metrics["precision"]*100]
aug_vals = [augmented_metrics["recall"]*100, augmented_metrics["f1"]*100, augmented_metrics["precision"]*100]

x = np.arange(len(metrics_names))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width/2, base_vals, width, label="Baseline (Sem GAN)", color="#EF4444")
plt.bar(x + width/2, aug_vals, width, label="Aumentado (Com cGAN)", color="#10B981")

plt.ylabel("Porcentagem (%)")
plt.title("Impacto Diagnóstico do Data Augmentation Generativo na Classe Minoritária Rara", fontweight="bold")
plt.xticks(x, metrics_names, fontweight="bold")
plt.ylim(0, 105)
plt.legend(frameon=True)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for i in range(len(metrics_names)):
    plt.text(x[i] - width/2, base_vals[i] + 2, f"{base_vals[i]:.1f}%", ha="center", fontsize=10, fontweight="bold")
    plt.text(x[i] + width/2, aug_vals[i] + 2, f"{aug_vals[i]:.1f}%", ha="center", fontsize=10, fontweight="bold", color="darkgreen")

plt.tight_layout()
plt.show()


---
## 🏆 Parte 7: Plano de Melhoria Integrado e Desafios para Casa

### Síntese dos 5 Problemas Técnicos e Seus Impactos Clínicos/Diagnósticos:
1. **Desbalanceamento Severo de Classes:** Resulta em viés do classificador para a classe majoritária, colapso de sensibilidade e alta taxa de **Falsos Negativos** (amostras críticas liberadas sem diagnóstico precoce).
2. **Shortcut Learning & Artefatos de Aquisição:** A rede pode correlacionar marcas de lâmina, gradientes de iluminação ou tubos à patologia. *Solução:* Segmentação prévia da região de interesse celular.
3. **Data Leakage entre Splits:** Múltiplas imagens do mesmo paciente/lâmina em treino e teste causam memorização individual e métricas falsamente infladas. *Solução:* Divisão estrita no nível de paciente/espécime (*GroupKFold*).
4. **Alucinações e Colapso de Modos pelo Modelo Generativo:** GANs sem regularização podem sintetizar artefatos biologicamente implausíveis ou repetir padrões idênticos. *Solução:* Consistência de ciclo (CycleGAN) e validação morfológica.
5. **Métrica Enganosa (Acurácia Global):** Em bases desbalanceadas, alta acurácia mascara Recall nulo da condição rara. *Solução:* Uso obrigatório de Recall e Curvas Precision-Recall (AUPRC).

---
### 🛠️ Desafios Práticos Propostos:
- **Desafio 1 (CycleGAN - Project 9C):** Implementar a perda de consistência de ciclo $\mathcal{L}_{cyc} = \|F(G(x)) - x\|_1$ para traduzir entre modalidades de microscopia (ex: holográfica $\to$ campo claro) preservando a integridade morfológica das células.
- **Desafio 2 (Hiperparâmetros de Augmentation):** Variar o número de imagens sintéticas adicionadas (40, 80, 160) e plotar a curva de calibração entre Recall e Especificidade.
- **Desafio 3 (Focal Loss):** Substituir a CrossEntropyLoss por Focal Loss ($\gamma = 2.0$) combinada com data augmentation generativo para avaliar sinergia algorítmica.
